In [130]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

### PPI

In [131]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [132]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 5394670.30it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [133]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [134]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [135]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        
        # Extract toxicity information at drug level
        toxicity = drug.find('db:toxicity', ns)
        toxicity_text = toxicity.text if toxicity is not None else None
        
        # Extract categories (may contain toxicity classifications)
        categories = drug.findall('db:categories/db:category', ns)
        category_list = []
        for cat in categories:
            cat_name = cat.find('db:category', ns)
            if cat_name is not None:
                category_list.append(cat_name.text)
        categories_str = ', '.join(category_list) if category_list else None
        
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None,
                    'toxicity': toxicity_text,
                    'categories': categories_str
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None,
                        'toxicity': toxicity_text,
                        'categories': categories_str
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [136]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')
Drug_bank['gene'] = Drug_bank['gene'].str.upper()  # Convert gene names to uppercase for consistency

Found 17430 drugs in the DrugBank XML.


In [137]:
### drugbank statistics
print(f"Total unique drugs: {Drug_bank['drug'].nunique()}")
print(f"Total unique genes: {Drug_bank['gene'].nunique()}")
print(f"Total unique drug-gene interactions: {len(Drug_bank)}")

Total unique drugs: 9368
Total unique genes: 4149
Total unique drug-gene interactions: 23136


In [138]:
Drug_bank

,drug,polypeptide,gene,gene_description,action,specific_function,toxicity,categories
0,Lepirudin,Prothrombin,F2,Prothrombin,inhibitor,calcium ion binding,The acute toxicity of intravenous lepirudin wa...,"Amino Acids, Peptides, and Proteins, Anticoagu..."
1,Cetuximab,Epidermal growth factor receptor,EGFR,Epidermal growth factor receptor,binder,actin filament binding,The intravenous LD<sub>50</sub> is > 300 mg/kg...,"Amino Acids, Peptides, and Proteins, Antibodie..."
2,Cetuximab,Low affinity immunoglobulin gamma Fc region re...,FCGR3B,Low affinity immunoglobulin gamma Fc region re...,binder,GPI anchor binding,The intravenous LD<sub>50</sub> is > 300 mg/kg...,"Amino Acids, Peptides, and Proteins, Antibodie..."
3,Cetuximab,Complement C1q subcomponent subunit A,C1QA,Complement C1q subcomponent subunit A,binder,amyloid-beta binding,The intravenous LD<sub>50</sub> is > 300 mg/kg...,"Amino Acids, Peptides, and Proteins, Antibodie..."
4,Cetuximab,Complement C1q subcomponent subunit B,C1QB,Complement C1q subcomponent subunit B,binder,None,The intravenous LD<sub>50</sub> is > 300 mg/kg...,"Amino Acids, Peptides, and Proteins, Antibodie..."
...,...,...,...,...,...,...,...,...
23131,Benzgalantamine,Muscle nicotinic acetylcholine receptor,None,Muscle nicotinic acetylcholine receptor,allosteric modulator,extracellular ligand-gated monoatomic ion chan...,Symptoms of overdose are expected to be simila...,"Alkaloids, Alzheimer Disease, drug therapy, Am..."
23132,Zoxazolamine,Small conductance calcium-activated potassium ...,KCNN2,Small conductance calcium-activated potassium ...,activator,alpha-actinin binding,None,"Antigout Preparations, Antirheumatic Agents, B..."
23133,Megestrol,Progesterone receptor,PGR,Progesterone receptor,binder,ATPase binding,None,"Adrenal Cortex Hormones, Antineoplastic Agents..."
23134,AZACYCLONOL,Histamine H1 receptor,HRH1,Histamine H1 receptor,inhibitor,G protein-coupled serotonin receptor activity,None,None


### Genetic Results

In [139]:
### import data
### genes
hpv_positive_amplification_top_gene_df= pd.read_csv('Results/CNV results/HPV positive amplification top genes.csv')
hpv_positive_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_positive_deletion_top_gene_df= pd.read_csv('Results/CNV results/HPV positive deletion top genes.csv')
hpv_positive_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_positive_deletion_top_gene_df['amplification_count'] = hpv_positive_deletion_top_gene_df['deletion_count']
hpv_positive_genes = pd.concat([hpv_positive_amplification_top_gene_df, hpv_positive_deletion_top_gene_df], axis=0)
hpv_positive_genes =hpv_positive_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_negative_amplification_top_gene_df = pd.read_csv('Results/CNV results/HPV negative amplification top genes.csv')
hpv_negative_amplification_top_gene_df['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_deletion_top_gene_df = pd.read_csv('Results/CNV results/HPV negative deletion top genes.csv')
hpv_negative_deletion_top_gene_df['MUT_TYPE'] = 'DELETION'
hpv_negative_deletion_top_gene_df['amplification_count'] = hpv_negative_deletion_top_gene_df['deletion_count']
hpv_negative_genes = pd.concat([hpv_negative_amplification_top_gene_df, hpv_negative_deletion_top_gene_df], axis=0)
hpv_negative_genes =hpv_negative_genes[['gene_name', 'gene', 'Sample_size', 'amplification_count',
       'frequency_percentage', 'gistic_score', 'p_value', 'q_value',
       'significant', 'empirical_p_value', 'empirical_q_value',
       'empirical_significant', 'MUT_TYPE']]

hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')

### EHR results

In [140]:
Drug_bank[Drug_bank['drug']=="Polyethylene glycol"]

,drug,polypeptide,gene,gene_description,action,specific_function,toxicity,categories


In [141]:
ehr_hpv_pos_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_positive_ml_drug_xgb_results.csv')
ehr_hpv_neg_drugs = pd.read_csv('../1. EHR based drug repurposing/Results/ML analysis/hpv_negative_ml_drug_xgb_results.csv')

### replace "_aspirin" in ehr_hpv_neg_drugs with "_acetylsalicylic acid"
ehr_hpv_neg_drugs['feature'] = ehr_hpv_neg_drugs['feature'].str.replace('_aspirin', '_acetylsalicylic acid')
ehr_hpv_pos_drugs['feature'] = ehr_hpv_pos_drugs['feature'].str.replace('_sennosides, USP', '_sennosides')

### Functions

In [142]:
def connect_to_genes(gene_df, ppi = protein_interaction):
    ### connect genes to those in PPI
    ### ensure only strong interactions are considered
    gene_df['gene_name'] = gene_df['gene_name'].str.upper()
    ppi['Translated_protein_1'] = ppi['Translated_protein_1'].str.upper()
    ppi['Translated_protein_2'] = ppi['Translated_protein_2'].str.upper()
    
    ppi = ppi[ppi['combined_score']>=700].reset_index(drop=True)
    ppi_available_genes = set(ppi['Translated_protein_1']).union(set(ppi['Translated_protein_2']))
    #### get literature validated genes
    for i, row in tqdm(gene_df.iterrows(), total=gene_df.shape[0]):
        connected_genes = []
        if row['gene_name'] in ppi_available_genes:
            ### get all connected genes
            connected_genes = ppi[ppi['Translated_protein_1'] == row['gene_name']]['Translated_protein_2'].tolist()
            connected_genes += ppi[ppi['Translated_protein_2'] == row['gene_name']]['Translated_protein_1'].tolist()
            connected_genes = list(set(connected_genes))
        else:
            connected_genes = []
        gene_df.loc[i, 'connected_genes'] = ', '.join(connected_genes)
        gene_df.loc[i, 'num_connected_genes'] = len(connected_genes)

    return gene_df

def number_of_unique_connected_genes(gene_df):
    unique_connected_genes = set()
    for connected_genes in gene_df['connected_genes']:
        if pd.notna(connected_genes):
            unique_connected_genes.update(connected_genes.split(', '))

    return list(unique_connected_genes), len(set(unique_connected_genes))

In [143]:
def connect_direct_gene_to_drug(gene_df, drug_df = Drug_bank):
    ### return two df one explicity drug-gene each one or one with drug-gene conections grouped by gene, and one with drug-gene connections grouped by drug
    gene_drug_connections = []
    for i, gene in tqdm(enumerate(gene_df['gene_name'])):
        connected_drugs = drug_df[drug_df['gene'] == gene]['drug'].tolist()
        gene_drug_connections.append({
            'gene': gene,
            'connected_drugs': ', '.join(connected_drugs),
            'num_connected_drugs': len(connected_drugs)
        })
    gene_drug_df = pd.DataFrame(gene_drug_connections)  
    # Create explicit drug-gene connections DataFrame
    explicit_df = []
    for _, row in gene_df.iterrows():
        gene = row['gene_name']
        connected_drugs = drug_df[drug_df['gene'] == gene]
        if len(connected_drugs) > 0:
            for _, drug_row in connected_drugs.iterrows():
                explicit_df.append({
                    'gene': gene,
                    'drug': drug_row['drug'],
                    'polypeptide': drug_row['polypeptide'],
                    'gene_description': drug_row['gene_description'],
                    'action': drug_row['action'],
                    'specific_function': drug_row['specific_function'],
                    'toxicity': drug_row['toxicity'],
                    'categories': drug_row['categories']
                })
    
    explicit_df = pd.DataFrame(explicit_df)
    # Merge with original gene_df to include mutation information
    explicit_df = explicit_df.merge(gene_df[['gene_name', 'MUT_TYPE', 'q_value', 'empirical_q_value']], 
                                     left_on='gene', right_on='gene_name', how='left')
    
    return explicit_df, gene_drug_df

In [144]:
def connect_indirect_gene_to_drug(indirect_gene_df, drug_df = Drug_bank):
    ### after running the connect_to_genes function, we can then connect the indirectly connected genes to drugs using the drug bank data
    ### return two df one with drug-indirect gene - risk gene explictly mentioned and one with drug-indirect gene - risk gene implicitly mentioned
    ### retrun andother df with grouped by drug with all the connected genes and risk genes mentioned
    # Extract all unique indirectly connected genes from the indirect_gene_df
    indirect_gene_df['gene_name'] = indirect_gene_df['gene_name'].str.upper()
    indirect_gene_df['connected_genes'] = indirect_gene_df['connected_genes'].str.upper()
    drug_df['gene'] = drug_df['gene'].str.upper()
    
    
    all_indirect_genes = []
    risk_gene_mapping = {}  # Map indirect gene to list of risk genes
    for idx, row in indirect_gene_df.iterrows():
        risk_gene = row['gene_name']
        connected_genes_str = row['connected_genes']
        if pd.notna(connected_genes_str) and connected_genes_str:
            connected_genes_list = [g.strip() for g in connected_genes_str.split(',')]
            all_indirect_genes.extend(connected_genes_list)
            # Map each indirect gene to its risk gene(s)
            for indirect_gene in connected_genes_list:
                if indirect_gene not in risk_gene_mapping:
                    risk_gene_mapping[indirect_gene] = []
                risk_gene_mapping[indirect_gene].append(risk_gene)

    # Get unique indirect genes
    unique_indirect_genes = list(set(all_indirect_genes))
    # Connect indirect genes to drugs
    drug_indirect_gene_risk_explicit = []
    for indirect_gene in tqdm(unique_indirect_genes):
        # Find drugs targeting this indirect gene
        connected_drugs = drug_df[drug_df['gene'] == indirect_gene]
        if len(connected_drugs) > 0:
            for _, drug_row in connected_drugs.iterrows():
                # Get all risk genes associated with this indirect gene
                
                associated_risk_genes = risk_gene_mapping.get(indirect_gene, [])
                for risk_gene in associated_risk_genes:
                    drug_indirect_gene_risk_explicit.append({
                        'drug': drug_row['drug'],
                        'indirect_gene': indirect_gene,
                        'risk_gene': risk_gene,
                        'polypeptide': drug_row['polypeptide'],
                        'gene_description': drug_row['gene_description'],
                        'action': drug_row['action'],
                        'specific_function': drug_row['specific_function'],
                        'toxicity': drug_row['toxicity'],
                        'categories': drug_row['categories']
                    })

    # Create DataFrame with explicit mapping
    df_explicit = pd.DataFrame(drug_indirect_gene_risk_explicit)

    # Create grouped DataFrame by drug
    if len(df_explicit) > 0:
        df_grouped = df_explicit.groupby('drug').agg({
            'indirect_gene': lambda x: ', '.join(sorted(set(x))),
            'risk_gene': lambda x: ', '.join(sorted(set(x))),
            'polypeptide': 'first',
            'action': 'first',
            'toxicity': 'first',
            'categories': 'first'
        }).reset_index()
        
        df_grouped.rename(columns={
            'indirect_gene': 'all_indirect_genes',
            'risk_gene': 'all_risk_genes'
        }, inplace=True)
    else:
        df_grouped = pd.DataFrame()

    return df_explicit, df_grouped

In [145]:
def overlap_ehr_drugs(gene_drug_df, ehr_drug_df):
    ### return df with merged ehr data on gene data only keeping ehr drugs that overlap with gene_drug_df
    ehr_drug_df = ehr_drug_df.rename(columns={'feature': 'drug'})
    ### remove'_' from drug names in ehr_drug_df if there
    ehr_drug_df['drug'] = ehr_drug_df['drug'].str.replace('_', '')
    ### case match both gene_drug_df and ehr_drug_df drug names to upper case
    gene_drug_df['drug'] = gene_drug_df['drug'].str.upper()
    ehr_drug_df['drug'] = ehr_drug_df['drug'].str.upper()
    overlap_df = gene_drug_df.merge(ehr_drug_df, left_on='drug', right_on='drug', how='inner', suffixes=('_gene', '_ehr'))
    ### print number of drugs validated
    print(f"Number of overlapped medications {len(set(overlap_df['drug']))}")
    print(f"Out of {len(set(ehr_drug_df['drug']))} medications in EHR data")
    print(f"Overlapped medications are: {list(set(overlap_df['drug']))}")
    return overlap_df

In [146]:
def connect_ehr_and_genetic_indirect(ehr_drug_df, genetic_gene_df):
    ### fix case sensitivity issue
    ehr_drug_df['feature'] = ehr_drug_df['feature'].str.upper()
    genetic_gene_df['drug'] = genetic_gene_df['drug'].str.upper()
    combined_results = []
    for i, row in ehr_drug_df.iterrows():
        drug_name = row['feature'].lstrip('_') # remove the _ prefix
        drug_name = drug_name.upper()  # Normalize to uppercase for matching
        # Check if this drug is connected to any genes in the genetic results
        for j, gene_row in genetic_gene_df.iterrows():
            if drug_name == gene_row['drug']:
                combined_results.append({
                    'drug': drug_name,
                    'gene_drug': gene_row['drug'],
                    'indirect_gene' : gene_row['indirect_gene'],
                    'risk_gene' : gene_row['risk_gene'],
                    'polypeptide': gene_row['polypeptide'],
                    'gene_description': gene_row['gene_description'],
                    'action': gene_row['action'],
                    'specific_function': gene_row['specific_function'],
                    'toxicity': gene_row['toxicity'],
                    'categories': gene_row['categories'],
                    'xgb_importance': row['xgb_importance'],
                    'xgb_absolute_importance': row['xgb_absolute_Importance'],
                    'rf_importance': row['rf_importance'],
                    'univariate_hr': row['univariate_hr'],
                    'multivariate_hr': row['multivariate_hr']
                    # 'genetic_additional_info': gene_row.to_dict()
                })
    overlap_df = pd.DataFrame(combined_results)
    print(f"Number of overlapped medications {len(set(overlap_df['drug']))}")
    print(f"Out of {len(set(ehr_drug_df['feature']))} medications in EHR data")
    print(f"Overlapped medications are: {list(set(overlap_df['drug']))}")
    return overlap_df

In [147]:
import plotly.graph_objects as go

def plot_sankey_drug_to_neighbor_to_riskgene(indirect_explicit_df, drug_name):
    """
    Plots a Sankey diagram for a given drug showing connections:
    drug → indirect_gene (neighbor) → risk_gene
    Ensures each node appears only once. Colors: blue (drug), green (neighbor), red (risk gene).
    """
    # Filter for the selected drug (case-insensitive)
    df = indirect_explicit_df[indirect_explicit_df['drug'].str.upper() == drug_name.upper()]
    if df.empty:
        print(f"No data for drug: {drug_name}")
        return

    # Unique node labels
    drug_label = drug_name.upper()
    indirect_genes = sorted(df['indirect_gene'].unique())
    risk_genes = sorted(df['risk_gene'].unique())

    # Build node list: drug, indirect genes, risk genes
    node_labels = [drug_label] + indirect_genes + risk_genes
    node_indices = {label: i for i, label in enumerate(node_labels)}

    # Links: drug → indirect_gene
    links_1 = df.groupby(['drug', 'indirect_gene']).size().reset_index(name='count')
    # Links: indirect_gene → risk_gene
    links_2 = df.groupby(['indirect_gene', 'risk_gene']).size().reset_index(name='count')

    # Sankey link lists
    sources = []
    targets = []
    values = []
    # drug → indirect_gene
    for _, row in links_1.iterrows():
        sources.append(node_indices[drug_label])
        targets.append(node_indices[row['indirect_gene']])
        values.append(row['count'])
    # indirect_gene → risk_gene
    for _, row in links_2.iterrows():
        sources.append(node_indices[row['indirect_gene']])
        targets.append(node_indices[row['risk_gene']])
        values.append(row['count'])

    # Colors: blue (drug), green (neighbor), red (risk gene)
    node_colors = ["#2196f3"] + ["#43a047"]*len(indirect_genes) + ["#e53935"]*len(risk_genes)

    # Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=node_labels,
            color=node_colors
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values
        )
    )])
    fig.update_layout(title_text=f"Sankey Diagram for {drug_label} (Drug → Neighbor → Risk Gene)", font_size=12)
    fig.show()

# Example usage:
# plot_sankey_drug_to_neighbor_to_riskgene(indirect_explicit, 'UREA')


In [148]:
from plotly.subplots import make_subplots

def plot_multiple_sankeys(indirect_explicit_df, drug_list, cols=2, title_prefix="", height=2200, width=2000, gene_targets=None, family_to_genes=None):
    """
    Plots multiple Sankey diagrams for a list of drugs in a grid layout (default 2 columns).
    Each subplot: drug → neighbor → risk gene (blue-green-red).
    Uniform node/link style for publication-quality consistency.
    """
    import plotly.graph_objects as go
    import math
    rows = math.ceil(len(drug_list) / cols)
    subplot_titles = [drug.title() for drug in drug_list]
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=subplot_titles,
        specs=[[{'type': 'sankey'}] * cols for _ in range(rows)],
        vertical_spacing=0.05,
        horizontal_spacing=0.03
    )

    for idx, drug in enumerate(drug_list):
        row = (idx // cols) + 1
        col = (idx % cols) + 1
        df = indirect_explicit_df[indirect_explicit_df['drug'].str.upper() == drug.upper()]
        if df.empty:
            continue
        drug_label = drug.upper()
        indirect_genes = sorted(df['indirect_gene'].unique())
        risk_genes = sorted(df['risk_gene'].unique())
        node_labels = [drug_label] + indirect_genes + risk_genes
        node_indices = {label: i for i, label in enumerate(node_labels)}
        # Links: drug → indirect_gene
        links_1 = df.groupby(['drug', 'indirect_gene']).size().reset_index(name='count')
        # Links: indirect_gene → risk_gene
        links_2 = df.groupby(['indirect_gene', 'risk_gene']).size().reset_index(name='count')
        sources = []
        targets = []
        values = []
        for _, row_link in links_1.iterrows():
            sources.append(node_indices[drug_label])
            targets.append(node_indices[row_link['indirect_gene']])
            values.append(1)  # Uniform thickness
        for _, row_link in links_2.iterrows():
            sources.append(node_indices[row_link['indirect_gene']])
            targets.append(node_indices[row_link['risk_gene']])
            values.append(1)  # Uniform thickness
        node_colors = []
        for label in node_labels:
            if label == drug_label:
                node_colors.append('rgba(31, 119, 180, 0.9)')  # Blue
            elif gene_targets and label in gene_targets:
                node_colors.append('rgba(44, 160, 44, 0.9)')   # Green
            elif label in indirect_genes:
                node_colors.append('rgba(44, 160, 44, 0.9)')   # Green
            else:
                node_colors.append('rgba(214, 39, 40, 0.9)')   # Red
        fig.add_trace(
            go.Sankey(
                arrangement='snap',
                valueformat='.0f',
                node=dict(
                    pad=8,
                    thickness=5,
                    line=dict(color="black", width=0.5),
                    label=node_labels,
                    color=node_colors
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    color='rgba(0, 0, 0, 0.2)'
                ),
                textfont=dict(size=15, color='black', family='Arial')
            ),
            row=row,
            col=col
        )
    fig.update_layout(
        title={
            'text': f"{title_prefix}Individual Drug Pathways<br>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 22}
        },
        height=height,
        width=width,
        showlegend=False,
        font=dict(size=14),
        margin=dict(t=100, b=20, l=20, r=20)
    )
    # Update subplot titles (drug names)
    for annotation in fig['layout']['annotations']:
        annotation['font'] = dict(size=18, color='black', family='Arial')
    # Add gene family legend if families are present
    if family_to_genes:
        all_gene_families = sorted(family_to_genes.keys())
        if all_gene_families:
            legend_lines = ["<b>Gene Families:</b>"]
            for family in all_gene_families:
                genes = family_to_genes[family]
                if len(genes) <= 3:
                    gene_list = ", ".join(genes)
                else:
                    gene_list = ", ".join(genes[:3]) + f", ... (+{len(genes)-3})"
                legend_lines.append(f"• <b>{family}</b>: {gene_list}")
            legend_text = "<br>".join(legend_lines)
            fig.add_annotation(
                text=legend_text,
                xref="paper", yref="paper",
                x=0.01, y=1,
                xanchor="left", yanchor="top",
                showarrow=False,
                font=dict(size=9, family="Arial"),
                align="left",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="black",
                borderwidth=1,
                borderpad=6
            )
    fig.show()

# Example usage:
# plot_multiple_sankeys(indirect_explicit, ['UREA', 'ARTENIMOL', ...], gene_targets=set(...), family_to_genes={...})


In [149]:
def group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap):
    """
    Group `hpv_pos_indirect_ehr_overlap` by `drug`.

    - For `indirect_gene` and `risk_gene`: collect all values across rows,
      split comma/semicolon/pipe-separated entries, strip, deduplicate, sort, and join with ', '.
    - For importance/HR columns keep the first non-null value per drug:
      `xgb_importance`, `rf_importance`, `univariate_hr`, `multivariate_hr`.

    Returns a DataFrame with columns:
        ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    """
    import re
    import pandas as pd
    # Defensive: ensure expected columns exist
    expected = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    hpv_pos_indirect_ehr_overlap = hpv_pos_indirect_ehr_overlap[expected]

    # Group and aggregate
    grouped = hpv_pos_indirect_ehr_overlap.groupby('drug').agg({
        'indirect_gene': lambda s: ','.join(sorted(set(s.dropna().astype(str)))),
        'risk_gene': lambda s: ','.join(sorted(set(s.dropna().astype(str)))),
        'xgb_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'rf_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'univariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'multivariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
    }).reset_index()

    out_cols = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    result = grouped[out_cols]
    return result


### HPV+

#### Genes

In [150]:
hpv_positive_genes[hpv_positive_genes['gene_name'] == 'RFC4']

,gene_name,gene,Sample_size,amplification_count,frequency_percentage,gistic_score,p_value,q_value,significant,empirical_p_value,empirical_q_value,empirical_significant,MUT_TYPE
48,RFC4,ENSG00000163918.11,221,74,33.484163,0.302421,1.164244e-56,4.901385e-54,True,0.000999,0.011578,True,AMPLIFICATION


In [151]:
hpv_pos_som_genes = hpv_positive_som_genes['Gene'].unique()
print(f"Number of unique genes in HPV positive somatic mutation top genes: {len(hpv_pos_som_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_pos_som_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV positive somatic mutation top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_pos_som_genes = hpv_positive_som_genes['Gene'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_pos_som_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV positive somatic mutation top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv pos som genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_pos_som_genes))
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(immediate_neighbors_and_risk_in_drugbank)}")
print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

Number of unique genes in HPV positive somatic mutation top genes: 6
Number of overlapping genes between HPV positive somatic mutation top genes and DrugBank genes: 1
Overlapping genes: {'PIK3CA'}
Number of immediate neighbors of the top genes in the PPI network: 611
Number of immediate neighbors or key risk genes that are in DrugBank: 232
Immediate neighbors or key risk genes that are in DrugBank: {'FLT1', 'HNF1A', 'RHOA', 'NTRK3', 'CDK1', 'CHUK', 'HDAC4', 'PIK3R3', 'CDC42', 'SRC', 'CDK9', 'JAK3', 'PIK3C2B', 'REL', 'HSP90AB1', 'INPP5D', 'IKBKE', 'PTPN11', 'CAMK2A', 'TLR4', 'AXL', 'RORA', 'CASP8', 'CDK4', 'DDX5', 'PCNA', 'KAT2A', 'CAMK2B', 'ITGB1', 'MAPT', 'IGF1R', 'MDM2', 'PIP4K2C', 'RXRA', 'NCOA1', 'PIK3CB', 'OGA', 'ESR1', 'IKBKG', 'IL1B', 'UBB', 'JAK2', 'CEBPB', 'NFE2L2', 'EGF', 'CD19', 'PIP4K2B', 'HDAC1', 'HDAC2', 'PIK3CD', 'NGFR', 'JAK1', 'RELA', 'HCK', 'ACTB', 'GAPDH', 'NFATC1', 'XIAP', 'RIPK1', 'MAP2K1', 'HDAC3', 'CCND1', 'TYK2', 'PPARG', 'EGFR', 'SIRT2', 'FGFR2', 'NOD2', 'SIRT1

In [152]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,AC003035.1,DELETION,8.234936784924659e-31,0.0203434456037747
1,AC003035.2,DELETION,8.234936784924659e-31,0.0203434456037747
2,AC003657.1,DELETION,8.234936784924659e-31,0.0203434456037747
3,AC003666.1,DELETION,8.234936784924659e-31,0.0203434456037747
4,AC003669.1,DELETION,8.234936784924659e-31,0.0203434456037747
...,...,...,...,...
442,ZFX-AS1,DELETION,8.234936784924659e-31,0.0203434456037747
443,ZMAT3,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357
444,ZNF639,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357
445,ZNF750,SOMATIC,1.6071012482810222e-08,0.0365630103656301


In [217]:
hpv_positive_combined_genes[hpv_positive_combined_genes['gene_name'] == 'SOX2']

,gene_name,MUT_TYPE,q_value,empirical_q_value,connected_genes,num_connected_genes
398,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,"SOX1, FOXA1, POU3F2, CDK1, CHD7, ESRRB, TUBB3,...",170.0


#### Expand through PPI

In [154]:
hpv_positive_connected_genes = connect_to_genes(hpv_positive_combined_genes)

100%|██████████| 447/447 [00:11<00:00, 39.26it/s] 


In [155]:
hpv_positive_connected_genes.sort_values('num_connected_genes', ascending=False).head(20)

,gene_name,MUT_TYPE,q_value,empirical_q_value,connected_genes,num_connected_genes
181,EP300,SOMATIC,0.0472395868080887,0.0365630103656301,"TCF12, MAX, HDAC4, MYOCD, HSP90AB1, SETD1A, GA...",390.0
311,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0115776022868357, 0.0365630103656301","FLT1, FOXA1, RHOA, NTRK3, CBLB, GAB1, RRAS, PI...",196.0
398,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0115776022868357,"SOX1, FOXA1, POU3F2, CDK1, CHD7, ESRRB, TUBB3,...",170.0
278,MRPL47,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"RPL26L1, REELD1, MTG1, LYRM7, MRPL13, PSMG1, M...",167.0
174,EIF1AX,DELETION,8.234936784924659e-31,0.0203434456037747,"RPL26L1, RPS27, RPS19, RPL10A, MRPL13, MRPS12,...",160.0
365,RPL39L,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"RPL26L1, RPS27, RPL30, RPS19, FTSJ3, RPL10A, M...",144.0
115,ACTL6A,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ACTR6, MORF4L1, ACTL6B, H2BC7, ANP32E, H2AC6, ...",137.0
176,EIF2S3,DELETION,8.234936784924659e-31,0.0203434456037747,"RPL26L1, RPS27, EIF2B2, RPL30, RPS19, RPL10A, ...",136.0
332,RBBP7,DELETION,8.234936784924659e-31,0.0203434456037747,"SAP30, MORF4L1, SCML2, KDM5A, CBX3, PHF1, AEBP...",136.0
285,NDUFB5,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ATP5F1C, COX6A1, UQCRC2, COX6A2, ECSIT, MT-CO3...",109.0


In [156]:
number_of_unique_connected_genes(hpv_positive_connected_genes)

(['',
  'REELD1',
  'ADH7',
  'PRCP',
  'CHPT1',
  'POU3F2',
  'NPHP1',
  'MRPS25',
  'PINK1',
  'TUFM',
  'MYOCD',
  'DLGAP1',
  'DNAI1',
  'HSP90AB1',
  'FGG',
  'PNISR',
  'COP1',
  'SETBP1',
  'KPNA4',
  'DUX4',
  'TTR',
  'SQSTM1',
  'ALKBH5',
  'C1R',
  'SOAT2',
  'ENSP00000485615',
  'BTRC',
  'MASP2',
  'RPA2',
  'FRRS1L',
  'CBX8',
  'SNRPE',
  'RPL13A',
  'KAT7',
  'EIF2B4',
  'NCOA6',
  'CD19',
  'GNPAT',
  'GATA4',
  'MTERF4',
  'GABPA',
  'F2R',
  'GTPBP8',
  'TAF15',
  'TRMT10C',
  'MRPS28',
  'CLDN1',
  'FPR1',
  'MEF2D',
  'RPL9',
  'ARID2',
  'HCG_1984214',
  'MRPS18B',
  'APOBEC1',
  'CCR5',
  'AFM',
  'MBTPS1',
  'RNASEH2A',
  'RBBP8',
  'MCRS1',
  'IL2',
  'OCLN',
  'RSAD2',
  'RBM14',
  'RETN',
  'SHH',
  'ASF1A',
  'RARA',
  'BRMS1',
  'GNA12',
  'UQCRH',
  'RNF2',
  'METTL3',
  'SYP',
  'DNAJC3',
  'KSR1',
  'IGF1',
  'MRNIP',
  'DHRS3',
  'RPS11',
  'BDKRB1',
  'SMARCE1',
  'HSCB',
  'MAPK11',
  'RPS6KA5',
  'NAPA',
  'TRIM28',
  'CEP131',
  'CUL7',
  'EPN1',
  

#### Connect to drugs

In [157]:
hpv_pos_direct_gene_drug_connections_explicit, hpv_pos_direct_gene_drug_connections_grouped = connect_direct_gene_to_drug(hpv_positive_connected_genes)

447it [00:00, 1495.13it/s]


In [158]:
hpv_pos_direct_gene_drug_connections_explicit

,gene,drug,polypeptide,gene_description,action,specific_function,toxicity,categories,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,Chloroquine,Angiotensin-converting enzyme 2,Angiotensin-converting enzyme 2,modulator,carboxypeptidase activity,Patients experiencing an overdose may present ...,"Agents Causing Muscle Toxicity, Agents that pr...",ACE2,DELETION,8.234936784924659e-31,0.0203434456037747
1,ACE2,Hydroxychloroquine,Angiotensin-converting enzyme 2,Angiotensin-converting enzyme 2,modulator,carboxypeptidase activity,Patients experiencing an overdose may present ...,"Aminoquinolines, Anti-Infective Agents, Antima...",ACE2,DELETION,8.234936784924659e-31,0.0203434456037747
2,ACE2,SPP1148,Angiotensin-converting enzyme 2,Angiotensin-converting enzyme 2,None,carboxypeptidase activity,None,None,ACE2,DELETION,8.234936784924659e-31,0.0203434456037747
3,ACE2,Bromhexine,Angiotensin-converting enzyme 2,Angiotensin-converting enzyme 2,binder,carboxypeptidase activity,The oral LD50 of bromhexine in rats is 6 g/kg....,"Amines, Aniline Compounds, Cough and Cold Prep...",ACE2,DELETION,8.234936784924659e-31,0.0203434456037747
4,ACE2,ORE-1001,Angiotensin-converting enzyme 2,Angiotensin-converting enzyme 2,inhibitor,carboxypeptidase activity,None,"Amino Acids, Amino Acids, Branched-Chain, Amin...",ACE2,DELETION,8.234936784924659e-31,0.0203434456037747
...,...,...,...,...,...,...,...,...,...,...,...,...
96,TLR8,Afimetoran,Toll-like receptor 8,Toll-like receptor 8,antagonist,DNA binding,None,None,TLR8,DELETION,8.234936784924659e-31,0.0203434456037747
97,VEGFD,TG-100801,Vascular endothelial growth factor D,Vascular endothelial growth factor D,inhibitor,chemoattractant activity,None,Benzene Derivatives,VEGFD,DELETION,8.234936784924659e-31,0.0203434456037747
98,VEGFD,Squalamine,Vascular endothelial growth factor D,Vascular endothelial growth factor D,modulator,chemoattractant activity,None,"Angiogenesis Inhibitors, Angiogenesis Modulati...",VEGFD,DELETION,8.234936784924659e-31,0.0203434456037747
99,VEGFD,NM-3,Vascular endothelial growth factor D,Vascular endothelial growth factor D,inhibitor,chemoattractant activity,None,None,VEGFD,DELETION,8.234936784924659e-31,0.0203434456037747


In [159]:
print(f"Number of directly connected drugs: {len(set(hpv_pos_direct_gene_drug_connections_explicit['drug']))}")

Number of directly connected drugs: 86


In [160]:
hpv_pos_df_indirect_explicit, hpv_pos_df_indirect_grouped = connect_indirect_gene_to_drug(hpv_positive_connected_genes)

100%|██████████| 3105/3105 [00:02<00:00, 1415.42it/s]


In [161]:
print(f"Number of unique genes that are targetable: {len(set(hpv_pos_df_indirect_explicit['indirect_gene']))}")
print(f"Number of unique risk genes connected to drugs: {len(set(hpv_pos_direct_gene_drug_connections_explicit['gene']))}")
print(f"Number of unique risk genes or indirectly connected genes connected to drugs: {len(set(hpv_pos_df_indirect_explicit['risk_gene']).union(set(hpv_pos_df_indirect_explicit['indirect_gene'])).union(set(hpv_pos_direct_gene_drug_connections_explicit['gene'])))}")

Number of unique genes that are targetable: 960
Number of unique risk genes connected to drugs: 28
Number of unique risk genes or indirectly connected genes connected to drugs: 1053


In [162]:
hpv_pos_df_indirect_explicit

,drug,indirect_gene,risk_gene,polypeptide,gene_description,action,specific_function,toxicity,categories
0,NADH,ADH7,PNPLA4,All-trans-retinol dehydrogenase [NAD(+)] ADH7,All-trans-retinol dehydrogenase [NAD(+)] ADH7,None,alcohol dehydrogenase (NAD+) activity,"No reports of overdose, however, high doses of...","Adenine Nucleotides, Coenzymes, Dietary Supple..."
1,Ethanol,ADH7,PNPLA4,All-trans-retinol dehydrogenase [NAD(+)] ADH7,All-trans-retinol dehydrogenase [NAD(+)] ADH7,substrate,alcohol dehydrogenase (NAD+) activity,"Oral, rat LD<sub>50</sub>: 5628 mg/kg. Symptom...","Agents Causing Muscle Toxicity, Alcohols, Anti..."
2,Zinc,TUFM,EIF1AX,"Elongation factor Tu, mitochondrial","Elongation factor Tu, mitochondrial",None,GTP binding,According to the Toxnet database of the U.S. N...,Agents for Treatment of Hemorrhoids and Anal F...
3,Zinc,TUFM,MRPL47,"Elongation factor Tu, mitochondrial","Elongation factor Tu, mitochondrial",None,GTP binding,According to the Toxnet database of the U.S. N...,Agents for Treatment of Hemorrhoids and Anal F...
4,Guanosine-5'-Diphosphate,TUFM,EIF1AX,"Elongation factor Tu, mitochondrial","Elongation factor Tu, mitochondrial",None,GTP binding,None,None
...,...,...,...,...,...,...,...,...,...
14178,Epigallocatechin gallate,FASN,ADIPOQ,Fatty acid synthase,Fatty acid synthase,inhibitor,(3R)-hydroxyacyl-[acyl-carrier-protein] dehydr...,None,"Anticarcinogenic Agents, Antimutagenic Agents,..."
14179,Biochanin A,FASN,ADIPOQ,Fatty acid synthase,Fatty acid synthase,inhibitor,(3R)-hydroxyacyl-[acyl-carrier-protein] dehydr...,None,"Anticarcinogenic Agents, Antineoplastic Agents..."
14180,Morin,FASN,ADIPOQ,Fatty acid synthase,Fatty acid synthase,inhibitor,(3R)-hydroxyacyl-[acyl-carrier-protein] dehydr...,None,"Antioxidants, Benzopyrans, Biological Factors,..."
14181,Pyridoxal phosphate,GAD1,SST,Glutamate decarboxylase 1,Glutamate decarboxylase 1,cofactor,glutamate decarboxylase activity,None,"Alimentary Tract and Metabolism, Coenzymes, Di..."


In [163]:
print(f'Number of medications identified {len(set(hpv_pos_df_indirect_explicit['drug']))}')

Number of medications identified 4176


In [164]:
print(f'Total number of medications identified {len(set(hpv_pos_df_indirect_explicit['drug']).union(set(hpv_pos_direct_gene_drug_connections_explicit['drug'])))}')

Total number of medications identified 4179


#### EHR match

In [165]:
hpv_pos_direct_ehr_overlap = overlap_ehr_drugs(hpv_pos_direct_gene_drug_connections_explicit, ehr_hpv_pos_drugs)

Number of overlapped medications 0
Out of 23 medications in EHR data
Overlapped medications are: []


In [166]:
hpv_pos_indirect_ehr_overlap = connect_ehr_and_genetic_indirect(ehr_hpv_pos_drugs, hpv_pos_df_indirect_explicit)

Number of overlapped medications 12
Out of 23 medications in EHR data
Overlapped medications are: ['ACETAMINOPHEN', 'ATORVASTATIN', 'ENOXAPARIN', 'BACITRACIN', 'MORPHINE', 'LOSARTAN', 'HEPARIN', 'OXYMETAZOLINE', 'DEXAMETHASONE', 'LEVOTHYROXINE', 'CARVEDILOL', 'ONDANSETRON']


In [167]:
grouped_ehr_pos_results = group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap)

In [168]:
grouped_ehr_pos_results

,drug,indirect_gene,risk_gene,xgb_importance,rf_importance,univariate_hr,multivariate_hr
0,ACETAMINOPHEN,"PTGES3,TRPV1","DNAJB11,KNG1",0.008007,0.022836,0.913998,1.014340
1,ATORVASTATIN,"DPP4,HDAC2","ACE2,ACTL6A,EP300,RBBP7,SCML2,TBL1X",0.000000,0.006100,0.922930,0.953439
2,BACITRACIN,A2M,"AHSG,HRG,KNG1",0.011568,0.005931,0.906710,0.971391
3,CARVEDILOL,"ADRB2,HIF1A,NDUFC2","EP300,NDUFB5,SOX2,SST",0.000000,0.001492,0.984616,1.124477
4,DEXAMETHASONE,"ANXA1,NR1I2,NR3C1","EP300,SOX2,ZFX",0.008129,0.012207,0.903507,1.025742
5,ENOXAPARIN,"F10,F2,SERPINC1","AHSG,HRG,KNG1",0.040709,0.007917,0.924706,1.030797
6,HEPARIN,"F10,FGF1,FGF2,FGF4,FGFR1,FGFR2,FGFR4,PF4,SERPINC1","AHSG,ANOS1,HRG,KNG1,PIK3CA,RPS6KA3,SOX2,VEGFD",0.009594,0.012315,0.951951,0.987667
7,LEVOTHYROXINE,"THRA,THRB","EP300,PIK3CA",0.044148,0.007657,0.881404,0.966715
8,LOSARTAN,"AGTR1,EPOR","ACE2,KNG1",0.000000,0.001408,0.739901,0.845143
9,MORPHINE,"OPRD1,OPRM1","GNB4,KNG1,SST",0.007568,0.009964,0.855894,0.957553


In [169]:
grouped_ehr_pos_results.to_csv('Results/Final results/hpv_positive_ehr_genetic_overlap_grouped.csv', index=False)

In [170]:
hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)[['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']]

,drug,indirect_gene,risk_gene,xgb_importance,rf_importance,univariate_hr,multivariate_hr
0,LEVOTHYROXINE,THRA,EP300,0.044148,0.007657,0.881404,0.966715
2,LEVOTHYROXINE,THRB,EP300,0.044148,0.007657,0.881404,0.966715
3,LEVOTHYROXINE,THRB,PIK3CA,0.044148,0.007657,0.881404,0.966715
1,LEVOTHYROXINE,THRA,PIK3CA,0.044148,0.007657,0.881404,0.966715
8,ENOXAPARIN,F10,KNG1,0.040709,0.007917,0.924706,1.030797
...,...,...,...,...,...,...,...
59,ATORVASTATIN,HDAC2,EP300,0.000000,0.006100,0.922930,0.953439
57,LOSARTAN,EPOR,ACE2,0.000000,0.001408,0.739901,0.845143
56,LOSARTAN,AGTR1,KNG1,0.000000,0.001408,0.739901,0.845143
55,LOSARTAN,AGTR1,ACE2,0.000000,0.001408,0.739901,0.845143


In [171]:
ehr_risk_genes = hpv_pos_indirect_ehr_overlap['risk_gene'].unique()
hpv_positive_combined_genes[hpv_positive_combined_genes['gene_name'].isin(ehr_risk_genes)]

,gene_name,MUT_TYPE,q_value,empirical_q_value,connected_genes,num_connected_genes
113,ACE2,DELETION,8.234936784924659e-31,0.0203434456037747,"ALB, IGHV3-72, IL2, PRCP, CTSS, MAS1L, IFNG, N...",63.0
115,ACTL6A,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ACTR6, MORF4L1, ACTL6B, H2BC7, ANP32E, H2AC6, ...",137.0
119,AHSG,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ALB, MGP, AFP, F2, CP, RBP4, FGG, SPP1, LPA, A...",57.0
130,ANOS1,DELETION,8.234936784924659e-31,0.0203434456037747,"GNRH1, HS6ST1, FGF8, VCX3A, CHD7, PROKR2, PROK...",14.0
169,DNAJB11,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"PPP5C, HSPA9, ERP29, PDIA3, PSMG1, CCT4, CCT6A...",57.0
181,EP300,SOMATIC,0.0472395868080887,0.0365630103656301,"TCF12, MAX, HDAC4, MYOCD, HSP90AB1, SETD1A, GA...",390.0
203,GNB4,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"GNAT2, LPAR1, DRD2, MAPK13, KCNJ6, GNA15, PLCB...",101.0
220,HRG,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ALB, F2, APOH, F10, APOA5, FGG, LPA, APOA1, SE...",34.0
227,KNG1,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"PRCP, SRC, PENK, CHRM3, F10, GAL, FGG, FN1, BD...",93.0
285,NDUFB5,AMPLIFICATION,4.9013845356499375e-54,0.0115776022868357,"ATP5F1C, COX6A1, UQCRC2, COX6A2, ECSIT, MT-CO3...",109.0


In [172]:
hpv_pos_indirect_ehr_overlap['drug'].nunique()

12

In [173]:
hpv_pos_indirect_ehr_overlap['risk_gene'].unique()

array(['EP300', 'PIK3CA', 'AHSG', 'HRG', 'KNG1', 'GNB4', 'SST', 'VEGFD',
       'RPS6KA3', 'ANOS1', 'SOX2', 'ZFX', 'DNAJB11', 'ACE2', 'ACTL6A',
       'RBBP7', 'SCML2', 'TBL1X', 'NDUFB5'], dtype=object)

In [174]:
hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10]

array(['LEVOTHYROXINE', 'ENOXAPARIN', 'BACITRACIN', 'ONDANSETRON',
       'HEPARIN', 'OXYMETAZOLINE', 'DEXAMETHASONE', 'ACETAMINOPHEN',
       'MORPHINE', 'ATORVASTATIN'], dtype=object)

In [175]:
hpv_pos_indirect_ehr_overlap.to_csv('Results/Integrated results/hpv_positive_indirect_ehr_overlap.csv', index=False)

In [176]:
# top_10_pos_drugs = list(hpv_pos_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
# hpv_pos_indirect_ehr_overlap[hpv_pos_indirect_ehr_overlap['drug'].isin(top_10_pos_drugs)]

In [177]:
hpv_pos_indirect_ehr_overlap

,drug,gene_drug,indirect_gene,risk_gene,polypeptide,gene_description,action,specific_function,toxicity,categories,xgb_importance,xgb_absolute_importance,rf_importance,univariate_hr,multivariate_hr
0,LEVOTHYROXINE,LEVOTHYROXINE,THRA,EP300,Thyroid hormone receptor alpha,Thyroid hormone receptor alpha,agonist,chromatin DNA binding,LD<sub>50</sub>=20 mg/kg (orally in rat). Hype...,"Agents used to treat hypothyroidism, Amino Aci...",0.044148,0.044148,0.007657,0.881404,0.966715
1,LEVOTHYROXINE,LEVOTHYROXINE,THRA,PIK3CA,Thyroid hormone receptor alpha,Thyroid hormone receptor alpha,agonist,chromatin DNA binding,LD<sub>50</sub>=20 mg/kg (orally in rat). Hype...,"Agents used to treat hypothyroidism, Amino Aci...",0.044148,0.044148,0.007657,0.881404,0.966715
2,LEVOTHYROXINE,LEVOTHYROXINE,THRB,EP300,Thyroid hormone receptor beta,Thyroid hormone receptor beta,agonist,chromatin DNA binding,LD<sub>50</sub>=20 mg/kg (orally in rat). Hype...,"Agents used to treat hypothyroidism, Amino Aci...",0.044148,0.044148,0.007657,0.881404,0.966715
3,LEVOTHYROXINE,LEVOTHYROXINE,THRB,PIK3CA,Thyroid hormone receptor beta,Thyroid hormone receptor beta,agonist,chromatin DNA binding,LD<sub>50</sub>=20 mg/kg (orally in rat). Hype...,"Agents used to treat hypothyroidism, Amino Aci...",0.044148,0.044148,0.007657,0.881404,0.966715
4,ENOXAPARIN,ENOXAPARIN,F2,AHSG,Prothrombin,Prothrombin,inhibitor,calcium ion binding,The oral LD50 for enoxaparin in mice is >5000 ...,"Agents causing hyperkalemia, Anticoagulants, B...",0.040709,0.040709,0.007917,0.924706,1.030797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,ATORVASTATIN,ATORVASTATIN,DPP4,ACE2,Dipeptidyl peptidase 4,Dipeptidyl peptidase 4,inhibitor,aminopeptidase activity,The reported LD50 of oral atorvastatin in mice...,"Agents Causing Muscle Toxicity, Anticholestere...",0.000000,0.000000,0.006100,0.922930,0.953439
64,CARVEDILOL,CARVEDILOL,HIF1A,EP300,Hypoxia-inducible factor 1-alpha,Hypoxia-inducible factor 1-alpha,modulator,cis-regulatory region sequence-specific DNA bi...,Patients experiencing an overdose may present ...,"Adrenergic Agents, Adrenergic alpha-1 Receptor...",0.000000,0.000000,0.001492,0.984616,1.124477
65,CARVEDILOL,CARVEDILOL,HIF1A,SOX2,Hypoxia-inducible factor 1-alpha,Hypoxia-inducible factor 1-alpha,modulator,cis-regulatory region sequence-specific DNA bi...,Patients experiencing an overdose may present ...,"Adrenergic Agents, Adrenergic alpha-1 Receptor...",0.000000,0.000000,0.001492,0.984616,1.124477
66,CARVEDILOL,CARVEDILOL,NDUFC2,NDUFB5,NADH dehydrogenase [ubiquinone] 1 subunit C2,NADH dehydrogenase [ubiquinone] 1 subunit C2,inhibitor,NADH dehydrogenase (ubiquinone) activity,Patients experiencing an overdose may present ...,"Adrenergic Agents, Adrenergic alpha-1 Receptor...",0.000000,0.000000,0.001492,0.984616,1.124477


In [178]:
# drugs_interested = ['ACETAMINOPHEN', 'HEPARIN', 'LEVOTHYROXINE', 'DEXAMETHASONE', 'ATORVASTATIN', 'LOSARTAN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, list(grouped_ehr_pos_results.sort_values(by='xgb_importance')['drug'].unique()[:10]), title_prefix="HPV Positive - ")

In [179]:
drugs_interested = ['LEVOTHYROXINE', 'ENOXAPARIN', 'BACITRACIN', 'ONDANSETRON', 'HEPARIN', 'OXYMETAZOLINE', 'DEXAMETHASONE', 'ACETAMINOPHEN', 'MORPHINE', 'ATORVASTATIN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, drugs_interested, title_prefix="HPV Positive - ")

In [180]:
### not showing glucose as it is a common medication that may be used in various contexts and may not be specific to the disease context we are studying, 
# and it may also have a high importance score due to its widespread use rather than a specific biological connection to the disease. 
# Therefore, we can choose to exclude it from the visualization to focus on more relevant drug-gene interactions.
### it has high connections to genes but is not specific to the disease context and may not provide meaningful insights in the analysis, 
# so we can choose to exclude it from the visualization to focus on more relevant drug-gene interactions.
drugs_interested = ['ACETAMINOPHEN', 'HEPARIN', 'LEVOTHYROXINE', 'DEXAMETHASONE', 'ATORVASTATIN', 'LOSARTAN']
plot_multiple_sankeys(hpv_pos_df_indirect_explicit, drugs_interested, title_prefix="HPV Positive - ")

In [181]:
plot_sankey_drug_to_neighbor_to_riskgene(hpv_pos_df_indirect_explicit, 'MORPHINE')

In [182]:
overlap_ehr_drugs(hpv_pos_direct_gene_drug_connections_explicit, ehr_hpv_pos_drugs)

Number of overlapped medications 0
Out of 23 medications in EHR data
Overlapped medications are: []


,gene,drug,polypeptide,gene_description,action,specific_function,toxicity,categories,gene_name,MUT_TYPE,...,multivariate_coef,multivariate_hr,multivariate_se,multivariate_p,multivariate_ci_lower,multivariate_ci_upper,multivariate_p_fdr,multivariate_significant_fdr,multivariate_significant,log_odds_ratio


### HPV-

In [183]:
hpv_negative_genes

,gene_name,gene,Sample_size,amplification_count,frequency_percentage,gistic_score,p_value,q_value,significant,empirical_p_value,empirical_q_value,empirical_significant,MUT_TYPE
0,MIR944,ENSG00000216058.1,1327,488,36.774680,0.332049,1.204131e-220,3.649901e-216,True,0.000999,0.004362,True,AMPLIFICATION
1,TPRG1-AS2,ENSG00000230115.1,1327,488,36.774680,0.331874,1.204131e-220,3.649901e-216,True,0.000999,0.004362,True,AMPLIFICATION
2,MTAPP2,ENSG00000230077.1,1327,482,36.322532,0.327260,3.141842e-215,6.348930e-211,True,0.000999,0.004362,True,AMPLIFICATION
3,AC117434.1,ENSG00000238026.1,1327,481,36.247174,0.322173,2.483125e-214,3.010689e-210,True,0.000999,0.004362,True,AMPLIFICATION
4,TP63,ENSG00000073282.14,1327,481,36.247174,0.324297,2.483125e-214,3.010689e-210,True,0.000999,0.004362,True,AMPLIFICATION
...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,MED13P1,ENSG00000236131.1,1327,262,19.743783,0.084808,2.650161e-281,5.005007e-279,True,0.000999,0.019177,True,DELETION
318,AC007241.1,ENSG00000278391.1,1327,262,19.743783,0.084956,2.650161e-281,5.005007e-279,True,0.000999,0.019177,True,DELETION
319,ELOCP26,ENSG00000236599.1,1327,262,19.743783,0.084956,2.650161e-281,5.005007e-279,True,0.000999,0.019177,True,DELETION
320,OFD1P4Y,ENSG00000229406.1,1327,262,19.743783,0.084956,2.650161e-281,5.005007e-279,True,0.000999,0.019177,True,DELETION


In [184]:
hpv_neg_som_genes = hpv_negative_som_genes['Gene'].unique()
print(f"Number of unique genes in HPV negative somatic mutation top genes: {len(hpv_neg_som_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_neg_som_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV negative somatic mutation top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_neg_som_genes = hpv_negative_som_genes['Gene'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_neg_som_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV negative somatic mutation top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv neg som genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_neg_som_genes))
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(immediate_neighbors_and_risk_in_drugbank)}")
print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

Number of unique genes in HPV negative somatic mutation top genes: 166
Number of overlapping genes between HPV negative somatic mutation top genes and DrugBank genes: 29
Overlapping genes: {'RHOA', 'SI', 'GABRB3', 'PDHA2', 'EPHA7', 'PIK3CA', 'EPHA5', 'TGFBR2', 'SELP', 'CASP8', 'HRAS', 'GABRG1', 'KRT5', 'NOTCH1', 'HLA-A', 'GRM8', 'KCNB2', 'NFE2L2', 'REG1A', 'EPHA2', 'KCNT2', 'TP53', 'KEAP1', 'HLA-B', 'RAC1', 'GABRA1', 'GRM1', 'ADCY2', 'GRM3'}
Number of immediate neighbors of the top genes in the PPI network: 2589
Number of immediate neighbors or key risk genes that are in DrugBank: 856
Immediate neighbors or key risk genes that are in DrugBank: {'ADH7', 'ITGAL', 'PRKAA1', 'PRKAR2A', 'MAPKAP1', 'DYRK2', 'PDHA2', 'NEDD8', 'G6PD', 'NLRP3', 'GLUD1', 'IGKV1D-33', 'HSP90AB1', 'S100A4', 'TOP1', 'LGALS1', 'FN1', 'AXL', 'SCARB1', 'NCL', 'XPO1', 'COL4A4', 'RRM1', 'SLC25A4', 'PDE3A', 'HMOX2', 'TYR', 'MCL1', 'RHOB', 'PIP4K2C', 'PTGS2', 'GLB1', 'NCOA1', 'TKT', 'PCK2', 'OGA', 'MMP7', 'GRM2', 'RAMP2',

In [185]:
### combine hpv positive somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_negative_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
1,ABCC5-AS1,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
2,ABCF3,AMPLIFICATION,1.4051416817717392e-199,0.0043623451388343
3,AC006328.1,DELETION,5.227396253868678e-285,0.0191774659792392
4,AC006328.2,DELETION,5.227396253868678e-285,0.0191774659792392
...,...,...,...,...
885,ZNF835,SOMATIC,1.0196847272849249e-05,0.005960073448722
886,ZNF839P1,DELETION,3.6107825764116486e-286,0.0191774659792392
887,ZNF885P,DELETION,1.4255963792291006e-287,0.0191774659792392
888,ZNF886P,DELETION,1.4255963792291006e-287,0.0191774659792392


In [218]:
hpv_negative_combined_genes[hpv_negative_combined_genes['gene_name'] == 'BCL6']

,gene_name,MUT_TYPE,q_value,empirical_q_value,connected_genes,num_connected_genes
200,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043623451388343,"NCOR2, BCORL1, IL4, CCND2, HDAC4, SIRT1, TNFRS...",69.0


In [187]:
hpv_negative_som_genes[hpv_negative_som_genes['Gene'] == 'TP53']

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score,MUT_TYPE,gene_name,q_value,empirical_q_value
0,TP53,405,345,0.234511,0.199768,0.0,0.0,True,0.0001,0.00596,79.676674,18.685034,SOMATIC,TP53,0.0,0.00596


#### Expand through PPI

In [188]:
hpv_negative_connected_genes = connect_to_genes(hpv_negative_combined_genes)

100%|██████████| 890/890 [00:21<00:00, 42.01it/s] 


In [189]:
number_of_unique_connected_genes(hpv_negative_connected_genes)

(['',
  'AMOT',
  'WAPL',
  'POU2F3',
  'CARD9',
  'PRCP',
  'REELD1',
  'NRG2',
  'CHPT1',
  'PRKAR2A',
  'MAFF',
  'POU3F2',
  'PRKAA1',
  'MRPS25',
  'PINK1',
  'TUFM',
  'ADH7',
  'DLGAP1',
  'DNAI1',
  'TREML1',
  'HSP90AB1',
  'FGG',
  'PNISR',
  'ARHGAP20',
  'COP1',
  'LGALS1',
  'GAN',
  'TGIF2LX',
  'RTN4',
  'KPNA4',
  'DUX4',
  'TTR',
  'SQSTM1',
  'ALKBH5',
  'COL4A4',
  'C1R',
  'KLRD1',
  'PPP2R2B',
  'BTRC',
  'FZD8',
  'MASP2',
  'RPA2',
  'MAGEC2',
  'RNF7',
  'FRRS1L',
  'CBX8',
  'PTDSS2',
  'SNRPE',
  'PCK2',
  'TAF8',
  'RPL13A',
  'EIF2B4',
  'GRM2',
  'CDH7',
  'ATL3',
  'CDH3',
  'TSPY8',
  'NCOA6',
  'CYP2C18',
  'CD19',
  'GNPAT',
  'MTERF4',
  'GATA4',
  'GABPA',
  'F2R',
  'GTPBP8',
  'ATRIP',
  'TAF15',
  'TRMT10C',
  'MRPS28',
  'TNKS1BP1',
  'CLDN1',
  'PDZD2',
  'GGA3',
  'SELPLG',
  'RPL9',
  'ARID2',
  'HCG_1984214',
  'E2F6',
  'CDC42BPA',
  'SLC44A2',
  'POLR2M',
  'ACOX3',
  'MRPS18B',
  'APOBEC1',
  'CCR5',
  'AFM',
  'MED30',
  'RNASEH2A',
  'RBB

#### Connect to drugs

In [190]:
hpv_neg_direct_gene_drug_connections_explicit, hpv_neg_direct_gene_drug_connections_grouped = connect_direct_gene_to_drug(hpv_negative_connected_genes)

890it [00:00, 1345.43it/s]


In [191]:
hpv_neg_direct_gene_drug_connections_explicit

,gene,drug,polypeptide,gene_description,action,specific_function,toxicity,categories,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ABCC5,Curcumin,ATP-binding cassette sub-family C member 5,ATP-binding cassette sub-family C member 5,inhibitor,ABC-type xenobiotic transporter activity,"In an acute oral toxicity study in mouse, LD50...","Alkanes, Analgesics, Analgesics, Non-Narcotic,...",ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
1,ABCC5,Curcumin sulfate,ATP-binding cassette sub-family C member 5,ATP-binding cassette sub-family C member 5,None,ABC-type xenobiotic transporter activity,"In an acute oral toxicity study in mouse, LD50...","Cytochrome P-450 CYP1A2 Inhibitors, Cytochrome...",ABCC5,AMPLIFICATION,2.0584024908359247e-200,0.0043623451388343
2,ADCY2,Colforsin,Adenylate cyclase type 2,Adenylate cyclase type 2,None,adenylate cyclase activity,None,"Adjuvants, Immunologic, Anti-Asthmatic Agents,...",ADCY2,SOMATIC,0.0003377698987821,0.005960073448722
3,ADCY2,"2',5'-DIDEOXY-ADENOSINE 3'-MONOPHOSPHATE",Adenylate cyclase type 2,Adenylate cyclase type 2,None,adenylate cyclase activity,None,None,ADCY2,SOMATIC,0.0003377698987821,0.005960073448722
4,ADCY2,Aurothioglucose,Adenylate cyclase type 2,Adenylate cyclase type 2,None,adenylate cyclase activity,Overdose as a result of too rapid increases in...,"Antiinflammatory and Antirheumatic Products, A...",ADCY2,SOMATIC,0.0003377698987821,0.005960073448722
...,...,...,...,...,...,...,...,...,...,...,...,...
361,TP53,Siremadlin,Cellular tumor antigen p53,Cellular tumor antigen p53,inhibitor,14-3-3 protein binding,None,None,TP53,SOMATIC,0.0,0.005960073448722
362,TP53,Thymoquinone,Cellular tumor antigen p53,Cellular tumor antigen p53,inhibitor,14-3-3 protein binding,None,Experimental Unapproved Treatments for COVID-1...,TP53,SOMATIC,0.0,0.005960073448722
363,TP53,Nutlin-3,Cellular tumor antigen p53,Cellular tumor antigen p53,inhibitor,14-3-3 protein binding,None,Stereoisomerism,TP53,SOMATIC,0.0,0.005960073448722
364,TP53,COTI-2,Cellular tumor antigen p53,Cellular tumor antigen p53,modulator,14-3-3 protein binding,None,None,TP53,SOMATIC,0.0,0.005960073448722


In [192]:
hpv_neg_direct_gene_drug_connections_explicit[hpv_neg_direct_gene_drug_connections_explicit['drug']=='Acetylsalicylic acid']

,gene,drug,polypeptide,gene_description,action,specific_function,toxicity,categories,gene_name,MUT_TYPE,q_value,empirical_q_value
350,TP53,Acetylsalicylic acid,Cellular tumor antigen p53,Cellular tumor antigen p53,inducer,14-3-3 protein binding,**Lethal doses**\r\n\r\nAcute oral LD50 values...,"Acids, Carbocyclic, Agents causing angioedema,...",TP53,SOMATIC,0.0,0.005960073448722


In [193]:
hpv_neg_df_indirect_explicit, hpv_neg_df_indirect_grouped = connect_indirect_gene_to_drug(hpv_negative_connected_genes)

100%|██████████| 4666/4666 [00:03<00:00, 1379.10it/s]


In [194]:
hpv_neg_df_indirect_grouped

,drug,all_indirect_genes,all_risk_genes,polypeptide,action,toxicity,categories
0,(11S)-8-CHLORO-11-[1-(METHYLSULFONYL)PIPERIDIN...,FNTB,HRAS,Protein farnesyltransferase subunit beta,None,None,None
1,"(13R,15S)-13-METHYL-16-OXA-8,9,12,22,24-PENTAA...",CDK2,"CASP8, CDKN2A, CDKN2B, TP53",Cyclin-dependent kinase 2,None,None,None
2,(1E)-5-(1-piperidin-4-yl-3-pyridin-4-yl-1H-pyr...,BRAF,"CDKN2A, FBXW7, HRAS, PIK3CA, TP53",Serine/threonine-protein kinase B-raf,None,None,None
3,(1H-indol-3-yl)-(2-mercapto-ethoxyimino)-aceti...,IL2,"HLA-A, TP53",Interleukin-2,None,None,None
4,"(1R)-1,2,2-trimethylpropyl (R)-methylphosphinate",CES1,"NCEH1, UGT2B4",Liver carboxylesterase 1,None,None,None
...,...,...,...,...,...,...,...
4958,tert-butanol,"CALM1, CALM3","ADCY2, ADCY8, HRAS, RAC1, RHOA",Calmodulin-3,inhibitor,None,"Alcohols, Butanols, Fatty Alcohols, Lipids"
4959,{(2S)-1-[N-(tert-butoxycarbonyl)glycyl]pyrroli...,F2,"AHSG, BCHE, HRG, KNG1",Prothrombin,None,None,None
4960,"{3-[3-(3,4-Dimethoxy-Phenyl)-1-(1-{1-[2-(3,4,5...",FKBP1A,RHOA,Peptidyl-prolyl cis-trans isomerase FKBP1A,None,None,None
4961,{4-[(2S)-2-Acetamido-3-({(1S)-1-[3-carbamoyl-4...,LCK,"NOTCH1, PIK3CA",Tyrosine-protein kinase Lck,None,None,None


In [195]:
hpv_neg_df_indirect_grouped[hpv_neg_df_indirect_grouped['drug']=='Dasatinib']

,drug,all_indirect_genes,all_risk_genes,polypeptide,action,toxicity,categories
1924,Dasatinib,"ABL1, BCR, BTK, EPHA2, EPHA5, EPHB4, FGR, FYN,...","CASP8, CDKN2A, CLDN1, DCC, DNAJB11, EIF4G1, EP...",Mast/stem cell growth factor receptor Kit,antagonist,Overdose cases with dasatinib occurred in isol...,"Antineoplastic Agents, Antineoplastic and Immu..."


#### EHR match

In [196]:
hpv_neg_direct_ehr_overlap = overlap_ehr_drugs(hpv_neg_direct_gene_drug_connections_explicit, ehr_hpv_neg_drugs)

Number of overlapped medications 1
Out of 36 medications in EHR data
Overlapped medications are: ['ACETYLSALICYLIC ACID']


In [197]:
hpv_neg_direct_ehr_overlap

,gene,drug,polypeptide,gene_description,action,specific_function,toxicity,categories,gene_name,MUT_TYPE,...,multivariate_coef,multivariate_hr,multivariate_se,multivariate_p,multivariate_ci_lower,multivariate_ci_upper,multivariate_p_fdr,multivariate_significant_fdr,multivariate_significant,log_odds_ratio
0,TP53,ACETYLSALICYLIC ACID,Cellular tumor antigen p53,Cellular tumor antigen p53,inducer,14-3-3 protein binding,**Lethal doses**\r\n\r\nAcute oral LD50 values...,"Acids, Carbocyclic, Agents causing angioedema,...",TP53,SOMATIC,...,-0.00647,0.993551,0.015216,0.670678,0.964358,1.023627,0.82013,False,False,0.616444


In [198]:
hpv_neg_indirect_ehr_overlap = overlap_ehr_drugs(hpv_neg_df_indirect_explicit, ehr_hpv_neg_drugs)

Number of overlapped medications 22
Out of 36 medications in EHR data
Overlapped medications are: ['ATORVASTATIN', 'LOSARTAN', 'TRAMADOL', 'FUROSEMIDE', 'METHYLPREDNISOLONE', 'POTASSIUM CHLORIDE', 'ACETAMINOPHEN', 'THIAMINE', 'ACETYLSALICYLIC ACID', 'IPRATROPIUM', 'METOPROLOL', 'LISINOPRIL', 'CLOPIDOGREL', 'LEVOTHYROXINE', 'PREDNISONE', 'ALBUTEROL', 'HYDRALAZINE', 'VILANTEROL', 'PRAVASTATIN', 'BUSPIRONE', 'MELATONIN', 'CARVEDILOL']


In [199]:
hpv_neg_indirect_ehr_overlap = connect_ehr_and_genetic_indirect(ehr_hpv_neg_drugs, hpv_neg_df_indirect_explicit)

Number of overlapped medications 22
Out of 36 medications in EHR data
Overlapped medications are: ['ATORVASTATIN', 'LOSARTAN', 'FUROSEMIDE', 'TRAMADOL', 'POTASSIUM CHLORIDE', 'ACETAMINOPHEN', 'THIAMINE', 'ACETYLSALICYLIC ACID', 'IPRATROPIUM', 'METOPROLOL', 'LISINOPRIL', 'CLOPIDOGREL', 'LEVOTHYROXINE', 'CARVEDILOL', 'PREDNISONE', 'ALBUTEROL', 'HYDRALAZINE', 'VILANTEROL', 'PRAVASTATIN', 'BUSPIRONE', 'MELATONIN', 'METHYLPREDNISOLONE']


In [200]:
hpv_neg_indirect_ehr_overlap

,drug,gene_drug,indirect_gene,risk_gene,polypeptide,gene_description,action,specific_function,toxicity,categories,xgb_importance,xgb_absolute_importance,rf_importance,univariate_hr,multivariate_hr
0,IPRATROPIUM,IPRATROPIUM,CHRM3,KNG1,Muscarinic acetylcholine receptor M3,Muscarinic acetylcholine receptor M3,antagonist,acetylcholine binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
1,IPRATROPIUM,IPRATROPIUM,CHRM2,GNB4,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
2,IPRATROPIUM,IPRATROPIUM,CHRM2,KNG1,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
3,IPRATROPIUM,IPRATROPIUM,CHRM2,SST,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
4,IPRATROPIUM,IPRATROPIUM,CHRM1,KNG1,Muscarinic acetylcholine receptor M1,Muscarinic acetylcholine receptor M1,antagonist,G protein-coupled acetylcholine receptor activity,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,BUSPIRONE,BUSPIRONE,DRD2,GNB4,D(2) dopamine receptor,D(2) dopamine receptor,antagonist,dopamine binding,The oral LD<sub>50</sub> of buspirone is 196 m...,"Agents that produce hypertension, Antidepressi...",0.000000,0.000000,0.000354,0.772028,0.755687
169,BUSPIRONE,BUSPIRONE,DRD2,SST,D(2) dopamine receptor,D(2) dopamine receptor,antagonist,dopamine binding,The oral LD<sub>50</sub> of buspirone is 196 m...,"Agents that produce hypertension, Antidepressi...",0.000000,0.000000,0.000354,0.772028,0.755687
170,BUSPIRONE,BUSPIRONE,DRD4,GNB4,D(4) dopamine receptor,D(4) dopamine receptor,antagonist,dopamine binding,The oral LD<sub>50</sub> of buspirone is 196 m...,"Agents that produce hypertension, Antidepressi...",0.000000,0.000000,0.000354,0.772028,0.755687
171,BUSPIRONE,BUSPIRONE,HTR1A,GNB4,5-hydroxytryptamine receptor 1A,5-hydroxytryptamine receptor 1A,partial agonist,G protein-coupled serotonin receptor activity,The oral LD<sub>50</sub> of buspirone is 196 m...,"Agents that produce hypertension, Antidepressi...",0.000000,0.000000,0.000354,0.772028,0.755687


In [201]:
top_10_neg_drugs = list(hpv_neg_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
hpv_neg_indirect_ehr_overlap_top_10_drugs = hpv_neg_indirect_ehr_overlap[hpv_neg_indirect_ehr_overlap['drug'].isin(top_10_neg_drugs)]
ehr_risk_genes = hpv_neg_indirect_ehr_overlap_top_10_drugs['risk_gene'].unique()
hpv_negative_combined_genes[hpv_negative_combined_genes['gene_name'].isin(ehr_risk_genes)]

,gene_name,MUT_TYPE,q_value,empirical_q_value,connected_genes,num_connected_genes
177,ACTL6A,AMPLIFICATION,4.6046768654042024e-207,0.0043623451388343,"ACTR6, MORF4L1, ACTL6B, H2BC7, ANP32E, H2AC6, ...",137.0
180,ADCY8,SOMATIC,0.0021845458816456,0.005960073448722,"PDE11A, GCH1, PDE2A, CAMK2A, PDE4B, PDE3A, CAM...",79.0
182,ADIPOQ,AMPLIFICATION,8.446185897603275e-199,0.0043623451388343,"PPARG, ALB, GCG, LEP, CIDEA, GPT, LPL, SIRT1, ...",63.0
187,AK5,SOMATIC,0.0120110141632539,0.005960073448722,"ADK, TPK1, NTPCR, CPNE6, ENTPD3, AK7, NT5C1B, ...",34.0
220,CDKN2A,"DELETION, SOMATIC","0.0, 2.518854531227791e-131","0.0191774659792392, 0.005960073448722","FHIT, CDK1, RRAS, CDC42, SRC, CDC45, ARF6, MAP...",146.0
250,CLDN16,AMPLIFICATION,2.2550778717096893e-195,0.0043623451388343,"CLDN25, OCLN, CLDN15, SLC12A3, CLDN18, CLDN5, ...",30.0
265,CTCF,SOMATIC,0.0022772779679226,0.005960073448722,"FOXA1, WAPL, ZBTB33, RBBP5, PARP1, TOP2A, MDFI...",69.0
279,DCUN1D1,AMPLIFICATION,8.470480026620086e-210,0.0043623451388343,"CUL2, DCUN1D4, UBE2F, NKX2-1, UBE2D3, NEDD8, C...",25.0
281,DNAJB11,AMPLIFICATION,8.446185897603275e-199,0.0043623451388343,"PPP5C, HSPA9, ERP29, PDIA3, PSMG1, CCT4, CCT6A...",57.0
342,FAT1,SOMATIC,1.1029271611301108e-39,0.005960073448722,"SCRIB, VASP, FJX1, NPHS2, MTNR1A, LAMC1, CTNNB...",10.0


In [202]:
ehr_risk_genes.sort()
ehr_risk_genes

array(['ACTL6A', 'ADCY8', 'ADIPOQ', 'AK5', 'CDKN2A', 'CLDN16', 'CTCF',
       'DCUN1D1', 'DNAJB11', 'FAT1', 'GABRA1', 'GNB4', 'GRM1', 'GRM3',
       'HLA-A', 'HLA-B', 'HRAS', 'KNG1', 'KRT5', 'MECOM', 'NFE2L2',
       'NOTCH1', 'P3H2', 'PCDH10', 'PHC3', 'PIK3CA', 'RAC1', 'SI', 'SOX2',
       'SST', 'TBL1XR1', 'TP53'], dtype=object)

In [203]:
hpv_neg_indirect_ehr_overlap.to_csv('Results/Integrated results/hpv_negative_indirect_ehr_overlap.csv', index=False)

In [204]:
grouped_ehr_neg_results = group_ehr_overlap_by_drug(hpv_neg_indirect_ehr_overlap)
### change "acetylsalicylic acid" to "aspirin"
grouped_ehr_neg_results['drug'] = grouped_ehr_neg_results['drug'].str.replace('ACETYLSALICYLIC ACID', 'ASPIRIN')

In [205]:
grouped_ehr_neg_results

,drug,indirect_gene,risk_gene,xgb_importance,rf_importance,univariate_hr,multivariate_hr
0,ACETAMINOPHEN,"PTGES3,PTGS2,TRPV1","CASP8,DNAJB11,KNG1,TP53",0.004543,0.019330,0.963788,0.995286
1,ASPIRIN,"AKR1C1,CASP1,CASP3,CCND1,EDNRA,HSPA5,IKBKB,MAP...","ACTL6A,BCL6,CASP8,CDKN2A,CDKN2B,CTCF,DCC,DNAJB...",0.003261,0.005344,0.926577,0.993551
2,ALBUTEROL,"ADRB2,ADRB3","ADIPOQ,SST",0.006685,0.006826,0.926996,0.979359
3,ATORVASTATIN,"DPP4,HDAC2","ACTL6A,MECOM,PHC3,SI,TBL1XR1,TP53",0.008874,0.007053,0.935884,0.996803
4,BUSPIRONE,"DRD2,DRD3,DRD4,HTR1A","GNB4,SST",0.000000,0.000354,0.772028,0.755687
5,CARVEDILOL,"ADRB2,HIF1A,NDUFC2,SELE,VCAM1","CDKN2A,DCUN1D1,NDUFB5,NFE2L2,NOTCH1,SELP,SOX2,...",0.000000,0.002189,0.927175,0.985951
6,CLOPIDOGREL,P2RY12,SELP,0.000000,0.001020,0.913138,1.029794
7,FUROSEMIDE,SLC12A1,CLDN16,0.008959,0.003794,0.862679,0.976657
8,HYDRALAZINE,"HIF1A,P4HA1","CDKN2A,DCUN1D1,NFE2L2,NOTCH1,P3H2,SOX2,TP53",0.006720,0.002854,0.895581,1.011587
9,IPRATROPIUM,"CHRM1,CHRM2,CHRM3","GNB4,KNG1,SST",0.014151,0.001699,0.916717,0.975212


In [206]:
grouped_ehr_neg_results[grouped_ehr_neg_results['drug'] == 'MELATONIN']['risk_gene'].to_list()

['ADCY8,ADIPOQ,CTCF,DNAJB11,FAT1,GNB4,GRM1,HLA-A,HLA-B,HRAS,KRT5,PIK3CA,RAC1,TP53']

In [207]:
grouped_ehr_neg_results[grouped_ehr_neg_results['drug'] == 'TRAMADOL']['indirect_gene'].to_list()

['ADORA1,CHRM1,CHRM3,GRIN1,OPRD1,OPRM1,SCN2A,TACR1,TRPV1']

In [208]:
list(grouped_ehr_neg_results['drug'].str.lower())

['acetaminophen',
 'aspirin',
 'albuterol',
 'atorvastatin',
 'buspirone',
 'carvedilol',
 'clopidogrel',
 'furosemide',
 'hydralazine',
 'ipratropium',
 'levothyroxine',
 'lisinopril',
 'losartan',
 'melatonin',
 'methylprednisolone',
 'metoprolol',
 'potassium chloride',
 'pravastatin',
 'prednisone',
 'thiamine',
 'tramadol',
 'vilanterol']

In [209]:
top_10_neg_drugs = list(hpv_neg_indirect_ehr_overlap.sort_values(by = 'xgb_importance', ascending=False)['drug'].unique()[:10])
hpv_neg_indirect_ehr_overlap[hpv_neg_indirect_ehr_overlap['drug'].isin(top_10_neg_drugs)]

,drug,gene_drug,indirect_gene,risk_gene,polypeptide,gene_description,action,specific_function,toxicity,categories,xgb_importance,xgb_absolute_importance,rf_importance,univariate_hr,multivariate_hr
0,IPRATROPIUM,IPRATROPIUM,CHRM3,KNG1,Muscarinic acetylcholine receptor M3,Muscarinic acetylcholine receptor M3,antagonist,acetylcholine binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
1,IPRATROPIUM,IPRATROPIUM,CHRM2,GNB4,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
2,IPRATROPIUM,IPRATROPIUM,CHRM2,KNG1,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
3,IPRATROPIUM,IPRATROPIUM,CHRM2,SST,Muscarinic acetylcholine receptor M2,Muscarinic acetylcholine receptor M2,antagonist,arrestin family protein binding,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
4,IPRATROPIUM,IPRATROPIUM,CHRM1,KNG1,Muscarinic acetylcholine receptor M1,Muscarinic acetylcholine receptor M1,antagonist,G protein-coupled acetylcholine receptor activity,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
5,IPRATROPIUM,IPRATROPIUM,CHRM1,SST,Muscarinic acetylcholine receptor M1,Muscarinic acetylcholine receptor M1,antagonist,G protein-coupled acetylcholine receptor activity,The reported LD50 in mice after oral administr...,"Adrenergics, Inhalants, Agents producing tachy...",0.014151,0.014151,0.001699,0.916717,0.975212
6,THIAMINE,THIAMINE,TPK1,AK5,Thiamine pyrophosphokinase 1,Thiamine pyrophosphokinase 1,substrate,ATP binding,Thiamine toxicity is uncommon; as excesses are...,"Alimentary Tract and Metabolism, Diet, Food, a...",0.010765,0.010765,0.002668,0.972506,1.012078
7,MELATONIN,MELATONIN,CALR,ADIPOQ,Calreticulin,Calreticulin,None,calcium ion binding,<p>Generally well-tolerated when taken orally....,"Amines, Antioxidants, Biogenic Amines, Biogeni...",0.010572,0.010572,0.003383,0.907962,0.976899
8,MELATONIN,MELATONIN,CALR,DNAJB11,Calreticulin,Calreticulin,None,calcium ion binding,<p>Generally well-tolerated when taken orally....,"Amines, Antioxidants, Biogenic Amines, Biogeni...",0.010572,0.010572,0.003383,0.907962,0.976899
9,MELATONIN,MELATONIN,CALR,HLA-A,Calreticulin,Calreticulin,None,calcium ion binding,<p>Generally well-tolerated when taken orally....,"Amines, Antioxidants, Biogenic Amines, Biogeni...",0.010572,0.010572,0.003383,0.907962,0.976899


In [210]:
grouped_ehr_neg_results.to_csv('Results/Integrated results/hpv_negative_grouped_ehr_overlap.csv', index=False)

In [211]:
interested_drugs_neg = ['IPRATROPIUM', 'THIAMINE', 'MELATONIN', 'PREDNISOLONE', 'TRAMADOL', 'FUROSEMIDE', 'ATORVASTATIN', 'HYDRALAZINE', 'ALBUTEROL', 'POTASSIUM CHLORIDE']
plot_multiple_sankeys(hpv_neg_df_indirect_explicit, interested_drugs_neg, title_prefix="HPV Negative - ")

In [212]:
interested_drugs_neg = ['METHYLPREDNISOLONE', 'MELATONIN', 'LEVOTHYROXINE', 'ACETAMINOPHEN', 'ATORVASTATIN', 'THIAMINE']
plot_multiple_sankeys(hpv_neg_df_indirect_explicit, interested_drugs_neg, title_prefix="HPV Negative - ")

In [213]:
plot_sankey_drug_to_neighbor_to_riskgene(hpv_neg_indirect_ehr_overlap, 'aspirin')

No data for drug: aspirin


In [214]:
def group_ehr_overlap_by_drug(hpv_pos_indirect_ehr_overlap):
    """
    Group `hpv_pos_indirect_ehr_overlap` by `drug`.

    - For `indirect_gene` and `risk_gene`: collect all values across rows,
      split comma/semicolon/pipe-separated entries, strip, deduplicate, sort, and join with ', '.
    - For importance/HR columns keep the first non-null value per drug:
      `xgb_importance`, `rf_importance`, `univariate_hr`, `multivariate_hr`.

    Returns a DataFrame with columns:
        ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    """
    import re
    import pandas as pd

    # Helper to flatten comma/semicolon/pipe separated strings in a Series
    def _flatten_and_join(series):
        parts = []
        for val in series.dropna().astype(str):
            # split on comma, semicolon or pipe
            for piece in re.split(r"[,;|]", val):
                p = piece.strip()
                if p:
                    parts.append(p)
        unique_sorted = sorted(set(parts))
        return ', '.join(unique_sorted)

    # Defensive: ensure expected columns exist
    expected = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    missing = [c for c in expected if c not in hpv_pos_indirect_ehr_overlap.columns]
    if missing:
        raise ValueError(f"Input DataFrame is missing required columns: {missing}")

    # Group and aggregate
    grouped = hpv_pos_indirect_ehr_overlap.groupby('drug').agg({
        # Keep `indirect_gene` joined as-is (no flattening/splitting)
        'indirect_gene': lambda s: ','.join(s.dropna().astype(str)),
        # For risk genes, still flatten/deduplicate/sort and join
        'risk_gene': lambda s: _flatten_and_join(s),
        'xgb_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'rf_importance': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'univariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
        'multivariate_hr': lambda s: (s.dropna().astype(float).iloc[0] if s.dropna().shape[0] > 0 else None),
    }).reset_index()

    out_cols = ['drug', 'indirect_gene', 'risk_gene', 'xgb_importance', 'rf_importance', 'univariate_hr', 'multivariate_hr']
    result = grouped[out_cols]
    return result
